Убираем все warnings:

In [ ]:
import warnings
warnings.filterwarnings('default') # ignore

Подключаем google disk для доступа к датасету и чекпоинтам:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
if os.path.exists("dataset_samples"):
    !rm -rf dataset_samples

Устанавливаем зависимости:

In [ ]:
!pip install datasets pyannote.metrics pyannote.audio huggingface_hub optuna onnxruntime onnxruntime-gpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.1 MB/s eta 0:00:00
 

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    print("Logging in HuggingFace...")
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Token has downloaded from Colab's secrets!")
    login(token=hf_token)
    print("Login successfully!")
except Exception as e:
    print(f"Error: {e}")
    hf_token = None

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected device: {DEVICE}")

In [ ]:
from pyannote.database import get_protocol, registry

config_path = '/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml'
registry.load_database(config_path)
protocol = get_protocol('AMI.SpeakerDiarization.word_and_vocalsounds')
files = list(protocol.development())
print(f"Loaded {len(files)} files from development subset")

In [ ]:
from pyannote.audio import Model

SEG_CKPT = "/content/drive/MyDrive/pyannote_finetuning/ami_segmentation_v1/fixed_version/checkpoints/last.ckpt"

segmentation_model = Model.from_pretrained(SEG_CKPT)
print("Segmentation model loaded directly from checkpoint")

Segmentation model loaded directly from checkpoint


In [ ]:
print(segmentation_model.hparams)

"linear":       {'hidden_size': 128, 'num_layers': 2}
"lstm":         {'hidden_size': 128, 'num_layers': 4, 'bidirectional': True, 'monolithic': True, 'dropout': 0.5, 'batch_first': True}
"num_channels": 1
"sample_rate":  16000
"sincnet":      {'stride': 10, 'sample_rate': 16000}


In [ ]:
from pyannote.audio import Pipeline
from pyannote.audio.utils.powerset import Powerset
from pyannote.audio.pipelines import SpeakerDiarization
from pyannote.audio.pipelines.clustering import AgglomerativeClustering

pretrained_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")

base_pipeline = SpeakerDiarization(
    segmentation=segmentation_model,
    embedding=pretrained_pipeline.embedding,
    embedding_exclude_overlap=pretrained_pipeline.embedding_exclude_overlap,
    clustering="AgglomerativeClustering"
)

In [ ]:
CORPUS_ROOT = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/"

pyannote_data = []
for file in files:
    uri = file['uri']
    audio_path = os.path.join(CORPUS_ROOT, f"amicorpus/{uri}/audio/{uri}.Mix-Headset.wav")

    if not os.path.exists(audio_path):
        print(f"Warning: {audio_path} not found, skipping")
        continue

    pyannote_data.append({
        "uri": uri,
        "audio": audio_path,
        "annotation": file['annotation'],
        "subset": None
    })

print(f"Prepared {len(pyannote_data)} files")

Выполняем подбор гиперпараметров с помощью библиотеки Optuna, которая является стандартом для оптимизации гиперпараметров в Pyannote:

In [ ]:
import optuna
import optuna.visualization as vis
from optuna.storages import RDBStorage
import time
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.audio import Pipeline

NUM_TRIALS = 50
DB_PATH = "/content/drive/MyDrive/optuna_study.sqlite3"

storage = RDBStorage(f"sqlite:///{DB_PATH}")
study_name = "my_speaker_diarization_study"

def objective(trial):
    print(f"=================================== Trial: {trial.number} ===================================")
    params = {
        "segmentation": {
            "min_duration_off": trial.suggest_float("min_duration_off", 0.0, 1.0),
        },
        "clustering": {
            "method": 'centroid',
            "min_cluster_size": trial.suggest_int("min_cluster_size", 2, 20),
            "threshold": trial.suggest_float("clustering_threshold", 0.1, 0.9),
        }
    }

    pipeline = base_pipeline.instantiate(params)
    pipeline.to(torch.device(DEVICE))
    metric = DiarizationErrorRate()
    for i, file in enumerate(pyannote_data):
        print(f"Processing file {i+1}/{len(pyannote_data)}: {file['uri']}")
        start = time.time()
        hypothesis = pipeline(file)
        hypothesis_diarization = hypothesis.speaker_diarization
        end = time.time()
        print(f"  Time: {end - start}s.")
        metric(file["annotation"], hypothesis_diarization)

    der = abs(metric)
    print(f"Trial {trial.number} DER = {der:.2%}")
    return der

try:
    study = optuna.load_study(study_name=study_name, storage=storage)
    print(f"Loaded existing study with {len(study.trials)} trials already completed.")
except KeyError:
    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
        direction="minimize",
    )
    print("Created new study.")
study.optimize(objective, n_trials=NUM_TRIALS)

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()
print(f"The best params: {study.best_params}")
print(f"The best DER: {study.best_value:.2%}")

Итого:

*   min_duration_off = 0.762
*   clustering_threshold = 0.36

DER = 20.62%